# PowerGrid Microgrid Balancing — GRPO Training

**Theme:** World Modeling — Professional Tasks (Theme 3.1)  
**Model:** Qwen2.5-1.5B-Instruct fine-tuned with GRPO via Unsloth + TRL  
**GPU:** T4 (free Colab) or A100  

### Reward signal — all components in [0, 1]
```
r_cost    = 1 - net_cost / max_step_cost   weight 0.60
r_solar   = solar_used / solar_available    weight 0.20
r_service = load_served / total_load        weight 0.10
r_stab    = soc_margin / 0.10              weight 0.10
reward    = 0.60*r_cost + 0.20*r_solar + 0.10*r_service + 0.10*r_stab
# Combined reward strictly in [0, 1] every step
```

In [ ]:
# Cell 1 — Install
!pip install -q unsloth trl transformers accelerate peft datasets numpy matplotlib
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' 2>/dev/null || true
print("Installation complete")

In [ ]:
# Cell 2 — Inline environment with [0,1] reward
import math, random, dataclasses
from dataclasses import dataclass

BATTERY_CAPACITY_KWH = 50.0
BATTERY_MAX_POWER_KW = 25.0
GRID_MAX_IMPORT_KW   = 50.0
BATTERY_EFFICIENCY   = 0.95
BATTERY_SOC_MIN      = 0.10
BATTERY_SOC_MAX      = 0.95
FLEXIBLE_LOAD_FRAC   = 0.30
GRID_EXPORT_RATIO    = 0.40
STEPS = 96; DT = 0.25
_MSC  = GRID_MAX_IMPORT_KW * 0.30 * DT   # 3.75 $/step

def _hour(t): return t * DT
def _solar(t, noise=0.0):
    h = _hour(t)
    if h < 6 or h > 18: return 0.0
    return max(0.0, 20.0 * math.exp(-((h-12)**2)/8) * (1 + noise*random.gauss(0,1)))
def _load(t, noise=0.0):
    h = _hour(t)
    return max(0.5, (3+8*math.exp(-((h-8)**2)/2)+12*math.exp(-((h-19)**2)/2))*(1+noise*random.gauss(0,0.5)))
def _price(t):
    h = _hour(t)
    return 0.30 if (7<=h<11 or 17<=h<21) else (0.15 if 11<=h<17 else 0.08)

@dataclass
class Obs:
    step:int; hour:float; solar_kw:float; demand_kw:float
    battery_soc:float; price:float; grid_import_kw:float
    episode_cost:float; done:bool; reward:float; context:str=''

class PowerGridEnvInline:
    def reset(self, seed=None):
        if seed is not None: random.seed(seed)
        self.t=0; self.soc=random.uniform(0.2,0.8); self.cost=0.0; self.done=False
        self._s=[_solar(t,0.05) for t in range(STEPS)]
        self._l=[_load(t,0.05)  for t in range(STEPS)]
        self._p=[_price(t)       for t in range(STEPS)]
        return self._obs(0.0,0.0)
    def step(self, bat, cur):
        bat=max(-1.0,min(1.0,float(bat))); cur=max(0.0,min(1.0,float(cur)))
        t=self.t; s,l,p=self._s[t],self._l[t],self._p[t]
        bk,self.soc=self._battery(bat*BATTERY_MAX_POWER_KW,self.soc)
        served=l-cur*l*FLEXIBLE_LOAD_FRAC; net=s-served-bk
        imp=max(0.0,-net); exp_=max(0.0,net)
        nc=imp*p*DT-exp_*p*GRID_EXPORT_RATIO*DT
        rc=max(0.0,min(1.0,1.0-nc/_MSC))
        rs=max(0.0,min(1.0,(s-exp_)/s)) if s>0.5 else 1.0
        rv=min(1.0,served/max(l,0.01))
        sm=min(self.soc-BATTERY_SOC_MIN,BATTERY_SOC_MAX-self.soc)
        rstab=max(0.0,min(1.0,sm/0.10))
        rew=0.60*rc+0.20*rs+0.10*rv+0.10*rstab
        self.cost+=max(0.0,nc); self.t+=1
        if self.t>=STEPS: self.done=True; rew=min(1.0,rew+0.05)
        return self._obs(imp,rew),rew,self.done,{}
    def _battery(self,desired,soc):
        if desired>0:
            mk=min(BATTERY_MAX_POWER_KW,(BATTERY_SOC_MAX-soc)*BATTERY_CAPACITY_KWH/(DT*BATTERY_EFFICIENCY))
            a=min(desired,mk); s2=soc+a*DT*BATTERY_EFFICIENCY/BATTERY_CAPACITY_KWH
        elif desired<0:
            mk=min(BATTERY_MAX_POWER_KW,(soc-BATTERY_SOC_MIN)*BATTERY_CAPACITY_KWH*BATTERY_EFFICIENCY/DT)
            a=max(desired,-mk); s2=soc-(-a)*DT/BATTERY_EFFICIENCY/BATTERY_CAPACITY_KWH
        else: a,s2=0.0,soc
        return a,max(BATTERY_SOC_MIN,min(BATTERY_SOC_MAX,s2))
    def _obs(self,imp,rew):
        t=min(self.t,STEPS-1); h=_hour(t); hi=int(h)
        p=self._p[t]; pl='ON-PEAK' if p>=0.30 else 'MID-PEAK' if p>=0.15 else 'OFF-PEAK'
        ctx=f'Time {hi:02d}:00 | Step {self.t}/96 | Solar {self._s[t]:.1f} kW | '\
            f'Demand {self._l[t]:.1f} kW | SoC {self.soc*100:.0f}% | '\
            f'Price {p:.2f} $/kWh ({pl}) | Cost ${self.cost:.2f}'
        return Obs(step=self.t,hour=h,solar_kw=self._s[t],demand_kw=self._l[t],
                   battery_soc=self.soc,price=p,grid_import_kw=imp,
                   episode_cost=self.cost,done=self.done,reward=rew,context=ctx)

_e=PowerGridEnvInline(); _o=_e.reset(seed=42); _o2,_r,_,_=_e.step(0.5,0.0)
assert 0.0<=_r<=1.0, f'Reward out of range: {_r}'
print(f"Env OK: step reward={_r:.4f} (in [0,1]), SoC={_o2.battery_soc:.2f}")

In [ ]:
# Cell 3 — Baselines
import numpy as np

def run_episode(env, policy_fn, seed=None):
    obs=env.reset(seed=seed); sr=[]
    while not obs.done:
        obs,r,_,_=env.step(*policy_fn(obs)); sr.append(r)
    return float(np.mean(sr)), obs.episode_cost

def random_policy(obs):       return random.uniform(-1,1), 0.0
def conservative_policy(obs): return 0.0, 0.0
def rule_based_policy(obs):
    h,soc=obs.hour,obs.battery_soc
    if 17<=h<21 and soc>0.3:        return -0.8,0.0
    if 10<=h<15 and obs.solar_kw>5: return  0.7,0.0
    if obs.price>=0.30 and soc>0.2: return -0.4,0.0
    return 0.0,0.0

env=PowerGridEnvInline(); N=20
rand_scores=[run_episode(env,random_policy,      seed=i)[0] for i in range(N)]
cons_scores=[run_episode(env,conservative_policy, seed=i)[0] for i in range(N)]
rule_scores=[run_episode(env,rule_based_policy,   seed=i)[0] for i in range(N)]
print(f"Random       {np.mean(rand_scores):.4f} +/- {np.std(rand_scores):.4f}")
print(f"Conservative {np.mean(cons_scores):.4f} +/- {np.std(cons_scores):.4f}")
print(f"Rule-based   {np.mean(rule_scores):.4f} +/- {np.std(rule_scores):.4f}")
print(f"GRPO target  > {max(np.mean(cons_scores), np.mean(rule_scores)):.4f}")

In [ ]:
# Cell 4 — Load Qwen2.5-1.5B with Unsloth (4-bit, fits on T4)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct", max_seq_length=1024,
    load_in_4bit=True, dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32, lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
print(model.print_trainable_parameters())

In [ ]:
# Cell 5 — Prompt template
SYSTEM = ('You are an expert power grid operator. Control a 50 kWh battery and flexible '
          'load to minimise electricity costs over 24 hours.\n'
          'Prices: OFF-PEAK $0.08 (21-07h), MID-PEAK $0.15 (11-17h), ON-PEAK $0.30 (07-11h,17-21h).\n'
          'Strategy: discharge ON-PEAK, charge from solar/OFF-PEAK, avoid curtailment.\n'
          'Respond ONLY with JSON: {"battery_action": <-1 to 1>, "curtailment": <0 to 1>>}')

def make_prompt(obs):
    sl=96-obs.step
    if   obs.price>=0.30 and obs.battery_soc>0.3:  h='ON-PEAK: discharge battery'
    elif obs.solar_kw>5  and obs.battery_soc<0.8:  h='Solar surplus: charge battery'
    elif obs.price<=0.08 and obs.battery_soc<0.6:  h='OFF-PEAK: charge battery cheaply'
    else:                                            h='Hold or small adjustment'
    return f'Grid state:\n{obs.context}\nSteps left: {sl}/96\nHint: {h}\nJSON action:'

def apply_chat_template(obs):
    msgs=[{"role":"system","content":SYSTEM},{"role":"user","content":make_prompt(obs)}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

env=PowerGridEnvInline(); obs=env.reset(seed=0)
print(apply_chat_template(obs)[:400])

In [ ]:
# Cell 6 — Action parser
import json, re

def parse_action(text):
    try:
        m=re.search(r'\{[^}]+\}', text, re.DOTALL)
        if m:
            d=json.loads(m.group())
            return max(-1.0,min(1.0,float(d.get("battery_action",0.0)))),\
                   max( 0.0,min(1.0,float(d.get("curtailment",   0.0))))
    except Exception: pass
    bm=re.search(r'battery_action["\s:]+([-\d.]+)', text)
    cm=re.search(r'curtailment["\s]+([\d.]+)',      text)
    return (max(-1.0,min(1.0,float(bm.group(1)))) if bm else 0.0,
            max( 0.0,min(1.0,float(cm.group(1)))) if cm else 0.0)

b,c=parse_action('{"battery_action":-0.8,"curtailment":0.0}')
print(f"Parse test: battery_action={b}, curtailment={c}")

In [ ]:
# Cell 7 — GRPO reward functions (both in [0,1])

def _step_reward(obs, batt, curt):
    p=obs.get('price',0.15); s=obs.get('solar_kw',0.0)
    soc=obs.get('battery_soc',0.5); l=obs.get('demand_kw',5.0)
    served=l-curt*l*FLEXIBLE_LOAD_FRAC
    net=s-served-batt*BATTERY_MAX_POWER_KW
    imp=max(0.0,-net); exp_=max(0.0,net)
    nc=imp*p*DT-exp_*p*GRID_EXPORT_RATIO*DT
    rc=max(0.0,min(1.0,1.0-nc/_MSC))
    rs=max(0.0,min(1.0,(s-exp_)/s)) if s>0.5 else 1.0
    rv=min(1.0,served/max(l,0.01))
    sm=min(soc-BATTERY_SOC_MIN,BATTERY_SOC_MAX-soc)
    rstab=max(0.0,min(1.0,sm/0.10))
    return 0.60*rc+0.20*rs+0.10*rv+0.10*rstab

def grpo_env_reward(prompts, completions, observations=None, **kwargs):
    obs_list=observations or [None]*len(completions)
    out=[]
    for comp,od in zip(completions,obs_list):
        text=comp[0]['content'] if isinstance(comp,list) else str(comp)
        bt,cu=parse_action(text)
        out.append(_step_reward(od,bt,cu) if od else 0.5)
    return out

def grpo_format_reward(prompts, completions, **kwargs):
    out=[]
    for comp in completions:
        text=comp[0]['content'] if isinstance(comp,list) else str(comp)
        r=0.0
        try:
            m=re.search(r'\{[^}]+\}',text,re.DOTALL)
            if m:
                d=json.loads(m.group()); r+=0.4
                if 'battery_action' in d and 'curtailment' in d:
                    r+=0.3
                    if -1<=float(d.get('battery_action',99))<=1: r+=0.2
                    if  0<=float(d.get('curtailment',99))<=1:    r+=0.1
        except Exception: pass
        out.append(min(1.0,r))
    return out

print("Reward functions OK — all outputs in [0, 1]")

In [ ]:
# Cell 8 — Training dataset (600 samples for better coverage)
from datasets import Dataset

def generate_training_data(n_episodes=60, steps_per_ep=10):
    data=[]; env=PowerGridEnvInline()
    for ep in range(n_episodes):
        obs=env.reset(seed=ep)
        for _ in range(random.randint(0,40)):
            if obs.done: break
            obs,_,_,_=env.step(*rule_based_policy(obs))
        for _ in range(steps_per_ep):
            if obs.done: break
            data.append({'prompt':apply_chat_template(obs),'obs':dataclasses.asdict(obs)})
            obs,_,_,_=env.step(*rule_based_policy(obs))
    return data

print("Generating training data...")
raw=generate_training_data(n_episodes=60, steps_per_ep=10)
dataset=Dataset.from_list(raw).shuffle(seed=42)
print(f"Dataset: {len(dataset)} samples"); print(dataset)


In [ ]:
# Cell 9 — GRPO Training (~20 min on T4)
import warnings
warnings.filterwarnings("ignore", message="Both .*and ")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.modeling_attn_mask_utils")
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir='./powergrid_grpo_checkpoint',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=256,
    max_completion_length=64,
    learning_rate=5e-6, warmup_steps=20,
    logging_steps=5, save_steps=50,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to='none',
    remove_unused_columns=False,
)

def reward_fn(prompts, completions, **kwargs):
    obs_list=kwargs.get('obs',[None]*len(prompts))
    env_r=grpo_env_reward(prompts, completions, observations=obs_list)
    fmt_r=grpo_format_reward(prompts, completions)
    return [0.7*e+0.3*f for e,f in zip(env_r,fmt_r)]

model.generation_config.max_length = None
trainer=GRPOTrainer(
    model=model, processing_class=tokenizer,
    reward_funcs=reward_fn, args=training_args, train_dataset=dataset,
)
print("Starting GRPO training (~20 min on T4)...")
train_result=trainer.train()
print("Done:", train_result.metrics)


In [ ]:
# Cell 10 — Evaluate trained LLM vs baselines
import warnings
warnings.filterwarnings("ignore", message="Both .*and ")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.modeling_attn_mask_utils")
FastLanguageModel.for_inference(model)
model.generation_config.max_length = None

def llm_policy(obs, temperature=0.1):
    inputs=tokenizer(apply_chat_template(obs),return_tensors='pt').to(model.device)
    with torch.no_grad():
        out=model.generate(**inputs,max_new_tokens=80,temperature=temperature,
                           do_sample=temperature>0,pad_token_id=tokenizer.eos_token_id)
    text=tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True)
    return parse_action(text)

env=PowerGridEnvInline(); N_EVAL=10
llm_scores=[run_episode(env,llm_policy,seed=i+100)[0] for i in range(N_EVAL)]
print(f"Random       {np.mean(rand_scores):.4f}")
print(f"Conservative {np.mean(cons_scores):.4f}")
print(f"Rule-based   {np.mean(rule_scores):.4f}")
print(f"GRPO LLM     {np.mean(llm_scores):.4f} +/- {np.std(llm_scores):.4f}")
imp=(np.mean(llm_scores)-np.mean(rand_scores))/max(np.mean(rand_scores),1e-9)*100
print(f"Improvement over random: {imp:+.1f}%")


In [ ]:
# Cell 11 — Results plots (save and commit powergrid_results.png)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('PowerGrid GRPO Training Results', fontsize=14, fontweight='bold')

# Plot 1: score comparison
ax=axes[0]
ags=['Random','Conservative','Rule-based','GRPO LLM']
scs=[np.mean(rand_scores),np.mean(cons_scores),np.mean(rule_scores),np.mean(llm_scores)]
ers=[np.std(rand_scores), np.std(cons_scores), np.std(rule_scores), np.std(llm_scores)]
bars=ax.bar(ags,scs,yerr=ers,capsize=5,
            color=['#e74c3c','#95a5a6','#f39c12','#27ae60'],alpha=0.85,edgecolor='black')
ax.set_ylim(0,1.0); ax.set_ylabel('Avg Step Score [0-1]')
ax.set_title('Score Comparison (higher = better)'); ax.grid(axis='y',alpha=0.4)
for bar,s in zip(bars,scs):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
            f'{s:.3f}',ha='center',fontsize=9,fontweight='bold')

# Plot 2: GRPO training curve
ax=axes[1]
try:
    logs=[x for x in trainer.state.log_history if 'reward' in x]
    st=[x['step'] for x in logs]; rw=[x['reward'] for x in logs]
    ax.plot(st,rw,color='steelblue',alpha=0.5,lw=1,label='Reward')
    if len(rw)>=5:
        sm=np.convolve(rw,np.ones(5)/5,mode='valid')
        ax.plot(st[2:-2],sm,color='crimson',lw=2,label='5-step MA')
    ax.set_ylim(0,1.0); ax.set_xlabel('Training step'); ax.set_ylabel('Reward [0-1]')
    ax.set_title('GRPO Training Curve'); ax.legend(); ax.grid(alpha=0.4)
except Exception:
    ax.text(0.5,0.5,'Run Cell 9 first',ha='center',va='center',transform=ax.transAxes)
    ax.set_title('GRPO Training Curve')

# Plot 3: 24h episode trace
ax=axes[2]; ax2=ax.twinx()
env2=PowerGridEnvInline(); obs=env2.reset(seed=42)
hrs,sol,dem,impt,soct=[],[],[],[],[]
while not obs.done:
    hrs.append(obs.hour); sol.append(obs.solar_kw); dem.append(obs.demand_kw)
    impt.append(obs.grid_import_kw); soct.append(obs.battery_soc*100)
    obs,_,_,_=env2.step(*llm_policy(obs))
ax.plot(hrs,sol,'y-',lw=2,label='Solar (kW)')
ax.plot(hrs,dem,'b--',lw=1.5,label='Demand (kW)')
ax.plot(hrs,impt,'r-',lw=1.5,label='Grid import (kW)')
ax2.plot(hrs,soct,'g-',lw=2,alpha=0.7,label='SoC (%)')
ax.set_xlabel('Hour of day'); ax.set_ylabel('Power (kW)')
ax2.set_ylabel('Battery SoC (%)',color='g')
ax.set_title('LLM Agent: 24-hour Episode')
l1,lb1=ax.get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
ax.legend(l1+l2,lb1+lb2,fontsize=7,loc='upper left'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('powergrid_results.png',dpi=150,bbox_inches='tight')
plt.show()
print("Saved powergrid_results.png — download and commit to repo")

In [ ]:
# Cell 12 — Save and push to HuggingFace Hub
HF_REPO = 'Ran1t/powergrid-grpo-qwen2.5-1.5b'
model.save_pretrained('powergrid_lora_model')
tokenizer.save_pretrained('powergrid_lora_model')
try:
    merged=model.merge_and_unload()
    merged.push_to_hub(HF_REPO, private=False)
    tokenizer.push_to_hub(HF_REPO, private=False)
    print(f"Model pushed: https://huggingface.co/{HF_REPO}")
except Exception as e:
    print(f"Push failed: {e}\nRun: huggingface-cli upload powergrid_lora_model")

## Results — fill in after running

| Agent | Avg Score [0-1] | Avg Cost ($/day) |
|---|---|---|
| Random | ~0.871 | ~$52 |
| Conservative | ~0.929 | ~$42 |
| Rule-based | ~0.900 | ~$40 |
| **GRPO LLM** | **TBD** | **TBD** |

After running: commit `powergrid_results.png` and update README.md with real numbers.